# 04 从候选池抽取未标记样本

本 notebook 不负责数据划分。它读取 03 生成的完整候选池，使用五个随机种子分别采样一次；每次都在每个“探针–蛋白质–残基类型”组内按 `NEGATIVE_RATIO` 抽取未标记位点。输出先按实验条件建立目录，再在每个数据集目录下保存五个 `seed_*.csv`。修改采样倍数时只需更改 `NEGATIVE_RATIO`，不需要重新运行 03。

In [5]:
# Cell 1：配置
import os
import random
import pandas as pd

DATASET_DIR = os.path.join("outputs", "dataset")
SAMPLED_ROOT = os.path.join("outputs", "sampled_data")
NEGATIVE_RATIO = 10

dataset_paths = {
    "Human_Lysate_100uM": os.path.join(DATASET_DIR, "Human", "Lysate", "100uM"),
    "Human_Lysate_1mM": os.path.join(DATASET_DIR, "Human", "Lysate", "1mM"),
    "SA59_Lysate_100uM": os.path.join(DATASET_DIR, "SA59", "Lysate", "100uM"),
    "SA59_Lysate_1mM": os.path.join(DATASET_DIR, "SA59", "Lysate", "1mM"),
    "SA59_Insitu_100uM": os.path.join(DATASET_DIR, "SA59", "Insitu", "100uM"),
}
SAMPLE_SEEDS = [42, 123, 2024, 3407, 8888]
EXPORT_COLUMNS = ["Probe", "Uniprot_ID", "Site", "Postion", "Label"]
RATIO_DIRNAME = f"1_{NEGATIVE_RATIO}"
os.makedirs(SAMPLED_ROOT, exist_ok=True)

In [6]:
# Cell 2：定义同蛋白、同残基类型的抽样函数
def sample_within_protein(positives, pool, ratio, seed):
    rng = random.Random(seed)
    pool_index = {
        (row.Probe, row.UNIPROT_ID, row.Site_type): tuple(map(int, row.Candidate_positions))
        for row in pool.itertuples(index=False)
    }

    negative_rows = []
    group_columns = ["Probe", "UNIPROT_ID", "Site_type"]
    for (probe, uid, site), group in positives.groupby(group_columns, sort=False):
        candidates = list(pool_index.get((probe, uid, site), ()))
        rng.shuffle(candidates)
        selected = candidates[:min(len(candidates), ratio * len(group))]
        negative_rows.extend(
            {"Probe": probe, "Uniprot_ID": uid, "Site": site, "Postion": position, "Label": 0}
            for position in selected
        )

    negatives = pd.DataFrame(negative_rows, columns=EXPORT_COLUMNS)
    observed = positives[["Probe", "UNIPROT_ID", "Site_type", "Position"]].rename(
        columns={"UNIPROT_ID": "Uniprot_ID", "Site_type": "Site", "Position": "Postion"}
    )
    observed["Label"] = 1
    sampled = pd.concat([observed[EXPORT_COLUMNS], negatives], ignore_index=True)
    assert not sampled.duplicated(["Probe", "Uniprot_ID", "Site", "Postion"]).any()
    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

In [7]:
# Cell 3：使用五个随机种子为每个条件生成五个采样版本
sampled_tables, summary_rows = {}, []
for seed in SAMPLE_SEEDS:
    for dataset_name, folder in dataset_paths.items():
        dataset_output_dir = os.path.join(SAMPLED_ROOT, dataset_name, RATIO_DIRNAME)
        os.makedirs(dataset_output_dir, exist_ok=True)
        positives = pd.read_csv(os.path.join(folder, "positive_sites.csv"))
        pool = pd.read_pickle(os.path.join(folder, "candidate_pool.pkl.gz"))
        sampled = sample_within_protein(positives, pool, NEGATIVE_RATIO, seed)
        sampled_tables[(dataset_name, seed)] = sampled
        output_path = os.path.join(dataset_output_dir, f"seed_{seed}.csv")
        sampled.to_csv(output_path, index=False, encoding="utf-8-sig")
        n_positive = int(sampled["Label"].sum())
        n_unlabeled = int((sampled["Label"] == 0).sum())
        summary_rows.append({
            "Dataset": dataset_name,
            "Seed": seed,
            "Configured_ratio": NEGATIVE_RATIO,
            "Positive_sites": n_positive,
            "Sampled_unlabeled_sites": n_unlabeled,
            "Actual_ratio": n_unlabeled / n_positive if n_positive else 0,
            "Output_file": output_path,
        })
        print(f"seed={seed}: {dataset_name}, positive={n_positive:,}, sampled={n_unlabeled:,}")

seed=42: Human_Lysate_100uM, positive=16,447, sampled=91,732
seed=42: Human_Lysate_1mM, positive=461, sampled=3,191
seed=42: SA59_Lysate_100uM, positive=41,991, sampled=165,265
seed=42: SA59_Lysate_1mM, positive=17,126, sampled=69,538
seed=42: SA59_Insitu_100uM, positive=6,503, sampled=33,094
seed=123: Human_Lysate_100uM, positive=16,447, sampled=91,732
seed=123: Human_Lysate_1mM, positive=461, sampled=3,191
seed=123: SA59_Lysate_100uM, positive=41,991, sampled=165,265
seed=123: SA59_Lysate_1mM, positive=17,126, sampled=69,538
seed=123: SA59_Insitu_100uM, positive=6,503, sampled=33,094
seed=2024: Human_Lysate_100uM, positive=16,447, sampled=91,732
seed=2024: Human_Lysate_1mM, positive=461, sampled=3,191
seed=2024: SA59_Lysate_100uM, positive=41,991, sampled=165,265
seed=2024: SA59_Lysate_1mM, positive=17,126, sampled=69,538
seed=2024: SA59_Insitu_100uM, positive=6,503, sampled=33,094
seed=3407: Human_Lysate_100uM, positive=16,447, sampled=91,732
seed=3407: Human_Lysate_1mM, positive=46

In [ ]:
# Cell 4：验证每组抽样上限并保存汇总
for (dataset_name, seed), sampled in sampled_tables.items():
    folder = dataset_paths[dataset_name]
    positives = pd.read_csv(os.path.join(folder, "positive_sites.csv"))
    pool = pd.read_pickle(os.path.join(folder, "candidate_pool.pkl.gz"))
    key = ["Probe", "UNIPROT_ID", "Site_type"]
    positive_counts = positives.groupby(key).size().to_dict()
    pool_counts = pool.set_index(key)["N_candidates"].to_dict()
    negatives = sampled[sampled["Label"].eq(0)].rename(
        columns={"Uniprot_ID": "UNIPROT_ID", "Site": "Site_type"}
    )
    negative_counts = negatives.groupby(key).size().to_dict()
    for group, n_positive in positive_counts.items():
        expected = min(NEGATIVE_RATIO * n_positive, int(pool_counts.get(group, 0)))
        assert negative_counts.get(group, 0) == expected

sampling_summary = pd.DataFrame(summary_rows)
sampling_summary.to_csv(os.path.join(SAMPLED_ROOT, f"sampling_summary_{RATIO_DIRNAME}.csv"), index=False, encoding="utf-8-sig")
sampling_summary

,Dataset,Seed,Configured_ratio,Positive_sites,Sampled_unlabeled_sites,Actual_ratio,Output_file
0,Human_Lysate_100uM,42,10,16447,91732,5.577431,outputs\sampled_data\Human_Lysate_100uM\1_10\s...
1,Human_Lysate_1mM,42,10,461,3191,6.921909,outputs\sampled_data\Human_Lysate_1mM\1_10\see...
2,SA59_Lysate_100uM,42,10,41991,165265,3.935724,outputs\sampled_data\SA59_Lysate_100uM\1_10\se...
3,SA59_Lysate_1mM,42,10,17126,69538,4.060376,outputs\sampled_data\SA59_Lysate_1mM\1_10\seed...
4,SA59_Insitu_100uM,42,10,6503,33094,5.089036,outputs\sampled_data\SA59_Insitu_100uM\1_10\se...
5,Human_Lysate_100uM,123,10,16447,91732,5.577431,outputs\sampled_data\Human_Lysate_100uM\1_10\s...
6,Human_Lysate_1mM,123,10,461,3191,6.921909,outputs\sampled_data\Human_Lysate_1mM\1_10\see...
7,SA59_Lysate_100uM,123,10,41991,165265,3.935724,outputs\sampled_data\SA59_Lysate_100uM\1_10\se...
8,SA59_Lysate_1mM,123,10,17126,69538,4.060376,outputs\sampled_data\SA59_Lysate_1mM\1_10\seed...
9,SA59_Insitu_100uM,123,10,6503,33094,5.089036,outputs\sampled_data\SA59_Insitu_100uM\1_10\se...


: 